In [4]:
import pandas as pd
from sqlalchemy import select

from models.app_models import Categorie, Produit
from utils.connections import app_session_factory

with app_session_factory() as app_session:
    query = select(Produit, Categorie.nom.label('categorie')).join(Produit.categorie, isouter=True)
    df = pd.read_sql(query, app_session.connection())

df

,produit_id,code,nom,prix_unitaire,actif,categorie_id,categorie
0,1,GL-VAN,Cornet vanille,2.8,True,1.0,Glaces
1,2,GL-CHO,Cornet chocolat,2.8,True,1.0,Glaces
2,3,GL-SPE,Cornet spéculoos,3.0,True,1.0,Glaces
3,4,GL-PIS,Cornet pistache,3.2,True,1.0,Glaces
4,5,GL-CUP,Coupe 3 boules,6.5,True,1.0,Glaces
5,6,SO-FRA,Sorbet fraise,2.8,True,2.0,Sorbets
6,7,SO-CIT,Sorbet citron,2.8,True,2.0,Sorbets
7,8,SO-MAN,Sorbet mangue,3.0,True,2.0,Sorbets
8,9,GA-LIE,Gaufre de Liège,3.5,True,3.0,Gaufres & crêpes
9,10,GA-BRU,Gaufre de Bruxelles,4.5,True,3.0,Gaufres & crêpes


In [5]:
import asyncio

import numpy as np

from utils.api import get_product_infos

df['nom'] = df['nom'].str.capitalize().str.strip()

# permet de d'envoyer toutes les requètes en même temps
new_columns = pd.DataFrame(await asyncio.gather(*[get_product_infos(row.nom) for row in df.itertuples()]))

for col in new_columns.columns:
    df[col] = new_columns[col]

cat_chaudes = ["Gaufres & crêpes", "Boissons chaudes"]
df["type_produit"] = np.where(df["categorie"].isin(cat_chaudes), "Chaud", "Froid")

df


,produit_id,code,nom,prix_unitaire,actif,categorie_id,categorie,allergenes,est_vegan,type_produit
0,1,GL-VAN,Cornet vanille,2.8,True,1.0,Glaces,"Lactose, Gluten, Fruits à coque",False,Froid
1,2,GL-CHO,Cornet chocolat,2.8,True,1.0,Glaces,"Lactose, Gluten, Fruits à coque",False,Froid
2,3,GL-SPE,Cornet spéculoos,3.0,True,1.0,Glaces,"Gluten, Lait, Fruits à coque",False,Froid
3,4,GL-PIS,Cornet pistache,3.2,True,1.0,Glaces,"Lait, Fruits à coques",False,Froid
4,5,GL-CUP,Coupe 3 boules,6.5,True,1.0,Glaces,"Lait, Fruits à coque",False,Froid
5,6,SO-FRA,Sorbet fraise,2.8,True,2.0,Sorbets,Peut contenir des traces de fruits à coque,True,Froid
6,7,SO-CIT,Sorbet citron,2.8,True,2.0,Sorbets,Aucun allergène courant n'est généralement pré...,True,Froid
7,8,SO-MAN,Sorbet mangue,3.0,True,2.0,Sorbets,Aucun,True,Froid
8,9,GA-LIE,Gaufre de liège,3.5,True,3.0,Gaufres & crêpes,"Blé, Lactose, Oeufs",False,Chaud
9,10,GA-BRU,Gaufre de bruxelles,4.5,True,3.0,Gaufres & crêpes,"Gluten, Lait, Oeufs",False,Chaud


In [6]:
from models.dw_models import DimProduit
from utils.connections import dw_session_factory

with dw_session_factory() as dw_session:
    df.drop(columns=["categorie_id"]).rename(columns={
        "prix_unitaire": "prix_catalogue",
        "actif": "est_actif",
    }).to_sql(
        DimProduit.__tablename__, 
        if_exists='append', 
        index=False, 
        con=dw_session.connection(),
    )
    dw_session.commit()